# Thai Air Intelligence — PM2.5 Regression + Classification

Native, cell-by-cell Google Colab workflow for the six requested regression
families and six matching multi-class classifiers. Classification labels are
derived only from the **actual next-day PM2.5 target**.

Run every cell from top to bottom in a fresh Colab runtime.

Required Colab Secrets: `SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY`.

Safe mode is the default: `REGISTER=False`, `ACTIVATE=False`. Review the dry-run
report and expanding-window evidence before registering inactive candidates.
Promotion is a separate explicit step after the database migration is verified.

In [ ]:
# 1. Configuration
REGISTER = False
ACTIVATE = False
APPROVED_CODE_SHA = "c4a564727d51f7581e2df3868f4613a25b8a1ec5"
PROVINCE = "all"  # "all" or one province such as "TH-30"
MINIMUM_ROWS = 180
CV_SPLITS = 5
SERVING_POLICY = "classifier_with_regression_fallback"
MODEL_FAMILIES = [
    "random_forest",
    "adaboost",
    "gradient_boosting",
    "xgboost",
    "lightgbm",
    "catboost",
]

if ACTIVATE and not REGISTER:
    raise ValueError("ACTIVATE=True requires REGISTER=True")
if len(APPROVED_CODE_SHA) != 40:
    raise ValueError("Set APPROVED_CODE_SHA to the reviewed 40-character commit SHA")

In [ ]:
# 2. Fetch reviewed code and install only packages missing from Colab
# Deliberately do not install/replace NumPy, Pandas, SciPy, or scikit-learn.
!rm -rf /content/THAI-AIR-INTELLIGENCE-LITE
!git clone --filter=blob:none https://github.com/kzabCde/THAI-AIR-INTELLIGENCE-LITE.git /content/THAI-AIR-INTELLIGENCE-LITE
!git -C /content/THAI-AIR-INTELLIGENCE-LITE checkout {APPROVED_CODE_SHA}
%cd /content/THAI-AIR-INTELLIGENCE-LITE
%pip install -q --upgrade-strategy only-if-needed   supabase==2.31.0   xgboost==2.1.4   lightgbm==4.6.0   catboost==1.2.8   statsmodels==0.14.4   joblib==1.4.2

In [ ]:
# 3. Scientific-stack ABI preflight
import platform
import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.ensemble import RandomForestRegressor

probe_X = np.asarray([[0.0], [1.0], [2.0], [3.0]], dtype=float)
probe_y = np.asarray([0.0, 1.0, 2.0, 3.0], dtype=float)
RandomForestRegressor(n_estimators=2, random_state=42).fit(probe_X, probe_y).predict(probe_X[:1])
pd.DataFrame({"abi_probe": probe_y}).describe()

print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
})
print("Environment ABI check passed")

In [ ]:
# 4. Load the two server-side Supabase secrets without printing their values
import os
from google.colab import userdata

for secret_name in ("SUPABASE_URL", "SUPABASE_SERVICE_ROLE_KEY"):
    secret_value = (userdata.get(secret_name) or "").strip()
    if not secret_value:
        raise ValueError(f"Missing Colab Secret: {secret_name}")
    os.environ[secret_name] = secret_value

if not os.environ["SUPABASE_URL"].startswith("https://"):
    raise ValueError("SUPABASE_URL must start with https://")
if os.environ["SUPABASE_SERVICE_ROLE_KEY"].lower().startswith(
    ("sb_publishable_", "sb_anon_")
):
    raise ValueError("Use a server-side service-role/secret key, not a public key")
print("Supabase secrets loaded")

In [ ]:
# 5. Import and validate the shared dual-model pipeline configuration
import json
import traceback
import uuid
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

from training.dual_model_config import FEATURE_COLUMNS, PipelineConfig
from training.pm25_classes import CLASS_IDS, THRESHOLD_VERSION, class_mapping
from training.train_dual_models import (
    PROVINCE_IDS,
    _json_safe,
    fetch_observed_rows,
    filter_training_rows,
    get_client,
    prepare_targets,
    register_and_maybe_activate,
    save_artifacts,
    summary_row,
    train_province,
)

config = PipelineConfig(
    minimum_rows=MINIMUM_ROWS,
    cv_splits=CV_SPLITS,
    serving_policy=SERVING_POLICY,
    artifact_directory=Path("training/artifacts"),
    allowed_model_families=tuple(MODEL_FAMILIES),
)
config.validate()
selected_provinces = (
    tuple(PROVINCE_IDS) if PROVINCE == "all" else (PROVINCE,)
)
if PROVINCE != "all" and PROVINCE not in PROVINCE_IDS:
    raise ValueError(f"Unknown province: {PROVINCE}")

print({
    "features": len(FEATURE_COLUMNS),
    "feature_order": list(FEATURE_COLUMNS),
    "model_families": MODEL_FAMILIES,
    "cv_splits": CV_SPLITS,
    "minimum_rows": MINIMUM_ROWS,
    "threshold_version": THRESHOLD_VERSION,
    "register": REGISTER,
    "activate": ACTIVATE,
})

In [ ]:
# 6. Connect to Supabase and fetch non-synthetic training data
sb = get_client()
raw = fetch_observed_rows(sb, selected_provinces)
print("Fetched rows:", len(raw))
print("Source: training_daily_summary_v2")
display(raw.head())

In [ ]:
# 7. Enforce open-meteo and trusted_hours >= 18; inspect quality by province
training_frame = filter_training_rows(raw, {"open-meteo"})
quality = (
    training_frame.groupby("province_id", as_index=False)
    .agg(
        usable_rows=("date", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        minimum_trusted_hours=("trusted_hours", "min"),
        mean_pm25=("pm25_mean", "mean"),
    )
    .sort_values("province_id")
)
quality["has_minimum_rows_before_target"] = quality["usable_rows"].ge(MINIMUM_ROWS)
display(quality)

In [ ]:
# 8. Build strict t -> t+1 targets and display actual next-day class distribution
prepared_by_province = {}
target_rows = []
class_rows = []

for province_id in selected_provinces:
    province_frame = training_frame[
        training_frame["province_id"] == province_id
    ].copy()
    prepared = prepare_targets(province_frame)
    prepared_by_province[province_id] = prepared
    target_rows.append({
        "province_id": province_id,
        "usable_target_rows": len(prepared),
        "first_feature_date": prepared["date"].min() if len(prepared) else None,
        "last_target_date": prepared["target_date"].max() if len(prepared) else None,
        "complete_feature_count": int(
            prepared.loc[:, FEATURE_COLUMNS].notna().all(axis=1).sum()
        ) if len(prepared) else 0,
    })
    counts = prepared["target_air_quality_class"].value_counts().to_dict()
    for class_id in CLASS_IDS:
        class_rows.append({
            "province_id": province_id,
            "class_id": class_id,
            "count": int(counts.get(class_id, 0)),
        })

target_quality = pd.DataFrame(target_rows)
class_distribution = pd.DataFrame(class_rows)
display(target_quality)
display(class_distribution.pivot(
    index="province_id", columns="class_id", values="count"
).fillna(0).astype(int))

In [ ]:
# 9. Preview chronological train/validation/final-holdout sizes
from training.train_dual_models import chronological_split

split_rows = []
for province_id, prepared in prepared_by_province.items():
    try:
        split = chronological_split(prepared, config)
        split_rows.append({
            "province_id": province_id,
            "train_rows": len(split.train_rows),
            "validation_rows": len(split.validation_rows),
            "final_holdout_rows": len(split.test_rows),
            "train_start": split.train_rows["date"].min(),
            "train_end": split.train_rows["date"].max(),
            "validation_start": split.validation_rows["date"].min(),
            "validation_end": split.validation_rows["date"].max(),
            "test_start": split.test_rows["date"].min(),
            "test_end": split.test_rows["date"].max(),
        })
    except Exception as exc:
        split_rows.append({
            "province_id": province_id,
            "error": f"{type(exc).__name__}: {exc}",
        })

split_audit = pd.DataFrame(split_rows)
display(split_audit)

In [ ]:
# 10. Train and compare 6 regressors + 6 classifiers independently per province
run_id = str(uuid.uuid4())
results = []
training_errors = []

for index, province_id in enumerate(selected_provinces, start=1):
    print(f"[{index}/{len(selected_provinces)}] Training {province_id}")
    try:
        result = train_province(
            province_id,
            training_frame[
                training_frame["province_id"] == province_id
            ].copy(),
            run_id,
            config,
        )
        save_artifacts(result, config.artifact_directory, config)
        results.append(result)
        print(json.dumps(summary_row(result), ensure_ascii=False))
    except Exception as exc:
        training_errors.append({
            "province_id": province_id,
            "error": type(exc).__name__,
            "message": str(exc),
            "traceback": traceback.format_exc(limit=5),
        })
        print(f"FAILED {province_id}: {type(exc).__name__}: {exc}")

print({
    "run_id": run_id,
    "successful_provinces": len(results),
    "failed_provinces": len(training_errors),
})
if not results:
    raise RuntimeError("No province completed training; inspect training_errors")

In [ ]:
# 11. Regression production-artifact metrics and selected teacher by province
regression_rows = []
for result in results:
    selected = result.regression
    regression_rows.append({
        "province_id": result.province_id,
        "teacher": selected.family,
        "teacher_mae": selected.teacher_test_metrics.get("mae"),
        "mae": selected.test_metrics.get("mae"),
        "rmse": selected.test_metrics.get("rmse"),
        "r2": selected.test_metrics.get("r2"),
        "skill_vs_persistence": selected.test_metrics.get(
            "skill_vs_persistence"
        ),
        "production_tuning": selected.tuning,
        "eligible": selected.eligible,
        "eligibility_reasons": ", ".join(selected.eligibility_reasons),
        "warnings": ", ".join(selected.warnings),
    })

regression_results = pd.DataFrame(regression_rows).sort_values("province_id")
display(regression_results)

In [ ]:
# 12. Classification production-artifact metrics and selected teacher by province
classification_rows = []
for result in results:
    selected = result.classification
    metrics = selected.test_metrics
    classification_rows.append({
        "province_id": result.province_id,
        "teacher": selected.family,
        "teacher_macro_f1": selected.teacher_test_metrics.get("macro_f1"),
        "accuracy": metrics.get("accuracy"),
        "balanced_accuracy": metrics.get("balanced_accuracy"),
        "macro_precision": metrics.get("macro_precision"),
        "macro_recall": metrics.get("macro_recall"),
        "macro_f1": metrics.get("macro_f1"),
        "production_tuning": selected.tuning,
        "eligible": selected.eligible,
        "eligibility_reasons": ", ".join(selected.eligibility_reasons),
        "warnings": ", ".join(selected.warnings),
    })

classification_results = pd.DataFrame(classification_rows).sort_values(
    "province_id"
)
display(classification_results)

In [ ]:
# 13. Precision, Recall, and F1 for every PM2.5 class
per_class_rows = []
for result in results:
    metrics_by_class = result.classification.test_metrics.get("per_class", {})
    for class_id in CLASS_IDS:
        values = metrics_by_class.get(str(class_id), {})
        per_class_rows.append({
            "province_id": result.province_id,
            "class_id": class_id,
            "precision": values.get("precision"),
            "recall": values.get("recall"),
            "f1": values.get("f1"),
            "support": values.get("support"),
        })

per_class_metrics = pd.DataFrame(per_class_rows)
display(per_class_metrics)

In [ ]:
# 14. Confusion matrix for each tuned production classifier
for result in results:
    matrix = result.classification.test_metrics.get("confusion_matrix", [])
    print(
        result.province_id,
        "production classifier distilled from teacher:",
        result.classification.family,
    )
    display(pd.DataFrame(
        matrix,
        index=[f"actual_{class_id}" for class_id in CLASS_IDS],
        columns=[f"predicted_{class_id}" for class_id in CLASS_IDS],
    ))

In [ ]:
# 15. Train/validation/test class distribution audit
audit_rows = []
for result in results:
    distributions = result.audit.get("class_distribution", {})
    for split_name, values in distributions.items():
        for class_id in CLASS_IDS:
            audit_rows.append({
                "province_id": result.province_id,
                "split": split_name,
                "class_id": class_id,
                "count": int(values.get(str(class_id), 0)),
            })

class_split_audit = pd.DataFrame(audit_rows)
display(class_split_audit)

In [ ]:
# 16. Eligibility and deterministic fallback plan
eligibility_rows = []
for result in results:
    eligibility_rows.append({
        "province_id": result.province_id,
        "regression_model": result.regression.family,
        "regression_eligible": result.regression.eligible,
        "classifier": result.classification.family,
        "classification_eligible": result.classification.eligible,
        "forecast_source": (
            "classifier"
            if result.classification.eligible
            else "regression_threshold"
            if result.regression.eligible
            else "persistence"
        ),
    })

eligibility = pd.DataFrame(eligibility_rows).sort_values("province_id")
display(eligibility)

In [ ]:
# 17. Province failures, warnings, and structured run summary
errors_df = pd.DataFrame(training_errors)
display(errors_df)

run_summary = {
    "run_id": run_id,
    "register": REGISTER,
    "activate": ACTIVATE,
    "serving_policy": SERVING_POLICY,
    "threshold_version": THRESHOLD_VERSION,
    "successful_provinces": len(results),
    "failed_provinces": len(training_errors),
    "results": [summary_row(result) for result in results],
    "errors": training_errors,
    "created_at": datetime.now(timezone.utc).isoformat(),
}
summary_path = config.artifact_directory / run_id / "run_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(
    json.dumps(_json_safe(run_summary), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("RUN_SUMMARY:", summary_path)

In [ ]:
# 18. Register inactive Regression + Classification candidates (guarded)
registration_status = []
if REGISTER:
    for result in results:
        try:
            register_and_maybe_activate(sb, result, activate=False)
            registration_status.append({
                "province_id": result.province_id,
                "registered": True,
                "error": None,
            })
        except Exception as exc:
            registration_status.append({
                "province_id": result.province_id,
                "registered": False,
                "error": f"{type(exc).__name__}: {exc}",
            })
    display(pd.DataFrame(registration_status))
else:
    print("REGISTER is False; no model_registry rows were written.")

In [ ]:
# 19. Activate only eligible task candidates (guarded and task-specific)
activation_status = []
if ACTIVATE:
    if not REGISTER:
        raise RuntimeError("ACTIVATE=True requires REGISTER=True")
    failed_registrations = {
        row["province_id"]
        for row in registration_status
        if not row["registered"]
    }
    for result in results:
        if result.province_id in failed_registrations:
            activation_status.append({
                "province_id": result.province_id,
                "activated": False,
                "error": "registration failed; activation skipped",
            })
            continue
        try:
            register_and_maybe_activate(sb, result, activate=True)
            activation_status.append({
                "province_id": result.province_id,
                "activated": True,
                "error": None,
            })
        except Exception as exc:
            activation_status.append({
                "province_id": result.province_id,
                "activated": False,
                "error": f"{type(exc).__name__}: {exc}",
            })
    display(pd.DataFrame(activation_status))
else:
    print("ACTIVATE is False; existing production active models are unchanged.")

In [ ]:
# 20. Verify at most one active model per province and task
active_rows = (
    sb.table("model_registry")
    .select(
        "province_id,task_type,model_name,model_family,run_id,"
        "eligibility_status,is_active"
    )
    .eq("is_active", True)
    .order("province_id")
    .order("task_type")
    .execute()
    .data
    or []
)
active_models = pd.DataFrame(active_rows)
display(active_models)
if not active_models.empty:
    active_counts = (
        active_models.groupby(["province_id", "task_type"])
        .size()
        .reset_index(name="active_count")
    )
    display(active_counts)
    assert active_counts["active_count"].max() <= 1

In [ ]:
# 21. Package auditable artifacts for download
import shutil
from google.colab import files

archive_base = Path(f"pm25_dual_artifacts_{run_id}")
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=config.artifact_directory / run_id,
)
print("ZIP ready:", archive_path)
files.download(archive_path)